# 06 – Final Submission (Phase 8)

**Before running:** tick every item in the AGENTS.md §12 pre-submission checklist.

**This notebook is reproducible.** Run "Restart & Run All" with internet off to confirm it produces `submission.csv` end-to-end.

In [ ]:
# ── Pre-submission checklist (AGENTS.md §12) ──────────────────────────────
CHECKLIST = {
    "All 303 test queries mapped to a category (≈3 per cat, no empty matches)": False,
    "Each row has exactly 20 distinct IDs, all in test_gallery.csv":             False,
    "Columns are exactly query_id,image_ids; IDs space-separated, best first":   False,
    "Ranking based on validated pipeline (config logged, seeds fixed)":          False,
    "No test labels / external images / food-specific checkpoints used":         False,
    "reports/models_used.md is complete":                                         False,
    "validate_submission.py passes":                                              False,
    "File saved with timestamp in artifacts/submissions/; log entry added":      False,
    "Final candidates selected by VALIDATION score and diversity, not LB":       False,
}
unchecked = [k for k, v in CHECKLIST.items() if not v]
if unchecked:
    print("UNCHECKED ITEMS (set to True before submitting):")
    for item in unchecked:
        print(f"  [ ] {item}")
else:
    print("All checklist items ticked. Proceeding...")

In [ ]:
import sys, pathlib
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, "..")
sys.path.append("/kaggle/input/datasets/somyajitsaha/food-retrieval-src")

from src.config import DATA_DIR, OUT_DIR, seed_everything
SEED = 42   # MUST match all prior notebooks
seed_everything(SEED)

import torchvision, timm, sklearn
try:
    import open_clip
    print("open_clip:", open_clip.__version__)
except ImportError:
    pass
print({k: v.__version__ for k, v in dict(
    torch=torch, torchvision=torchvision, timm=timm, sklearn=sklearn, numpy=np).items()})
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from src.data import load_csvs, parse_val_ground_truth
from src.features import load_features, load_text_features
from src.text_matching import build_surface_form_dict, TextMatcher, evaluate_text_matching
from src.ensemble import LogitEnsemble
from src.rerank import rerank
from src.metrics import evaluate, print_metrics
from src.submit import build_submission, validate_submission, save_submission, run_official_validator

dfs = load_csvs(DATA_DIR)
meta         = dfs["category_metadata"]
val_gallery  = dfs["val_gallery"]
val_queries  = dfs["val_queries"]
test_gallery = dfs["test_gallery"]
test_queries = dfs["test_queries"]
val_gt       = parse_val_ground_truth(dfs["val_gt"])

surface_forms = build_surface_form_dict(meta)
matcher       = TextMatcher(surface_forms, meta_df=meta)
cat_id_list   = meta["category_id"].tolist()
cat_id2idx    = {int(c): i for i, c in enumerate(cat_id_list)}

## 1. Verify text matcher on test queries

In [ ]:
test_mappings = [(str(row["query_id"]), *matcher.predict(str(row["description"])))
                 for _, row in test_queries.iterrows()]

unmatched = [(qid, desc) for (qid, cat_id, method, _), (_, row) in
             zip(test_mappings, test_queries.iterrows()) if cat_id == -1]
assert len(unmatched) == 0, f"{len(unmatched)} queries have no category match!"

# Verify ~3 queries per category
from collections import Counter
cat_counts = Counter(cat_id for _, cat_id, _, _ in test_mappings)
print(f"Queries per category – min={min(cat_counts.values())}, max={max(cat_counts.values())}")
print(f"Total test queries: {len(test_mappings)}")
print("All 303 test queries mapped. ✓")

## 2. Load best models and compute final test scores

In [ ]:
# ── Adjust MODEL_TAGS and temperatures to the best config from notebook 05 ─
MODEL_TAGS   = ["vitl14_openai", "siglip_so400m"]  # top models from Phase 3/4/5
TEMPERATURES = {"vitl14_openai": 1.0, "siglip_so400m": 1.0}  # from fit_temperature()
WEIGHTS      = {"vitl14_openai": 1.0, "siglip_so400m": 1.0}  # from ensemble search

test_scores = {}
test_feats  = {}

for tag in MODEL_TAGS:
    feat_dir = pathlib.Path(OUT_DIR) / "features" / tag
    X_t, test_ids = load_features(feat_dir, "test", gallery_df=test_gallery)
    T_t, _        = load_text_features(feat_dir, tag="text")
    test_scores[tag] = X_t @ T_t.T
    test_feats[tag]  = X_t

# Ensemble
ens = LogitEnsemble()
for tag in MODEL_TAGS:
    ens.add(test_scores[tag], weight=WEIGHTS[tag], temperature=TEMPERATURES[tag])
S_test = ens.combine()  # (N_test, K)

# Re-ranking (use best Sinkhorn strength from notebook 05)
SINKHORN_STRENGTH = 0.5   # replace with best value from notebook 05
avg_test_feats = np.mean([test_feats[t] for t in MODEL_TAGS], axis=0)
avg_test_feats /= np.linalg.norm(avg_test_feats, axis=1, keepdims=True) + 1e-9

S_final = rerank(S_test, image_features=avg_test_feats,
                 use_sinkhorn=True, sinkhorn_strength=SINKHORN_STRENGTH,
                 use_knn=False)

print(f"Final score matrix: {S_final.shape}")

## 3. Build and validate submission

In [ ]:
test_query_ids  = [str(r["query_id"]) for _, r in test_queries.iterrows()]
test_cat_indices = [cat_id2idx.get(cat_id, 0)
                    for _, cat_id, _, _ in test_mappings]

sub_df = build_submission(S_final, test_ids, test_query_ids, test_cat_indices, top=20)

is_valid = validate_submission(sub_df, test_gallery)
assert is_valid, "Internal validation FAILED – fix before submitting."

CONFIG_NAME = f"ensemble_{'+'.join(MODEL_TAGS)}_sink{SINKHORN_STRENGTH}"
sub_path = save_submission(sub_df, pathlib.Path(OUT_DIR) / "submissions", config_name=CONFIG_NAME)
print(f"Submission saved: {sub_path}")

In [ ]:
# Run official validator
passed = run_official_validator(sub_path, DATA_DIR)
assert passed, "Official validator FAILED – do not submit."

## 4. Log to experiment_log.md

In [ ]:
# Compute final validation score for the log
val_scores_dict = {}
for tag in MODEL_TAGS:
    feat_dir = pathlib.Path(OUT_DIR) / "features" / tag
    X_v, val_ids = load_features(feat_dir, "val", gallery_df=val_gallery)
    T_v, _       = load_text_features(feat_dir, tag="text")
    val_scores_dict[tag] = X_v @ T_v.T

ens_val = LogitEnsemble()
for tag in MODEL_TAGS:
    ens_val.add(val_scores_dict[tag], weight=WEIGHTS[tag], temperature=TEMPERATURES[tag])
S_val_final = ens_val.combine()

avg_val_feats = np.mean([load_features(pathlib.Path(OUT_DIR)/"features"/t, "val")[0] for t in MODEL_TAGS], axis=0)
avg_val_feats /= np.linalg.norm(avg_val_feats, axis=1, keepdims=True) + 1e-9
S_val_rr = rerank(S_val_final, image_features=avg_val_feats,
                  use_sinkhorn=True, sinkhorn_strength=SINKHORN_STRENGTH)

val_pred_dict = {}
for _, row in val_queries.iterrows():
    cat_id, _, _ = matcher.predict(str(row["description"]))
    cat_idx = cat_id2idx.get(cat_id, 0)
    order = np.argsort(-S_val_rr[:, cat_idx], kind="stable")
    val_pred_dict[str(row["query_id"])] = [val_ids[j] for j in order[:20]]

final_val_metrics = evaluate(val_pred_dict, val_gt)
print("Final validation metrics:")
print_metrics(final_val_metrics)

import pandas as pd
print("\nAdd to reports/experiment_log.md (Submissions section):")
print(f"| sub_final | {pd.Timestamp.now().date()} | {CONFIG_NAME} | {final_val_metrics['r_precision']:.4f} | — | YES | final candidate |")